# Análise Estatística: Uso de ChatGPT e Desempenho

**Shapiro-Wilk** (teste de normalidade, α = 0,05)


### 1. Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

### 2. Upload e leitura do CSV

In [ ]:
df = pd.read_csv("/content/CSV GERADO.csv", decimal=',')

df.columns = ['USOU_EMAIL', 'USOU_MEDIA', 'NAO_USOU_EMAIL', 'NAO_USOU_MEDIA', 'ELIMINADOS_EMAIL']

df.head()

,USOU_EMAIL,USOU_MEDIA,NAO_USOU_EMAIL,NAO_USOU_MEDIA,ELIMINADOS_EMAIL
0,vinicius.cmarques@ufrpe.br,5.83,joao.nunesneves@ufrpe.br,5.00,davi.coutinho@ufrpe.br
1,carla.rios@ufrpe.br,5.00,pedro.vsalves@ufrpe.br,4.17,lauro.aguiar@ufrpe.br
2,pedro.antonios@ufrpe.br,9.17,vinicius.henkenskk@ufrpe.br,3.33,adilsonc.1958@gmail.com
3,sostenes.dantas@ufrpe.br,4.17,douglas.bezerra@ufrpe.br,5.00,jean.arruda@ufrpe.br
4,joao.matheusmoura@ufrpe.br,5.00,edilmo.kaiky@ufrpe.br,7.50,alanemicaelle123@gmail.com


### 3. Separação dos Grupos

In [ ]:
usou = df['USOU_MEDIA'].dropna().to_numpy()
nao_usou = df['NAO_USOU_MEDIA'].dropna().to_numpy()
eliminados = df['ELIMINADOS_EMAIL'].dropna()

n_usou = len(usou)
n_nao_usou = len(nao_usou)
n_eliminados = len(eliminados)
n_total = n_usou + n_nao_usou + n_eliminados

print(f'Participantes que usaram ChatGPT: {n_usou}')
print(f'Participantes que não usaram ChatGPT: {n_nao_usou}')
print(f'Eliminados: {n_eliminados}')
print(f'Total de participantes: {n_total}')

Participantes que usaram ChatGPT: 24
Participantes que não usaram ChatGPT: 24
Eliminados: 9
Total de participantes: 57


### 4. Teste de normalidade (Shapiro-Wilk)

Usamos Shapiro-Wilk (e não o teste t, que não testa normalidade) com **α = 0,05**.

Critério: se **p ≥ 0,05**, não rejeitamos a hipótese de normalidade. Se **p < 0,05**, rejeitamos.

**Por que seguir com Mann-Whitney mesmo se a normalidade for confirmada?**

A amostra aqui é pequena (n = 24 por grupo). O Shapiro-Wilk tem pouco poder estatístico para detectar desvios de normalidade em amostras pequenas, então um resultado "normal" não garante totalmente que a distribuição real seja normal — pode ser apenas que a amostra não é grande o suficiente para o teste detectar o desvio.

Por isso, mesmo com a normalidade aferida no passo anterior, o Mann-Whitney é usado como abordagem mais conservadora: ele não assume normalidade e perde muito pouca eficiência estatística mesmo quando os dados são de fato normais (eficiência assintótica de ~95% em relação ao teste t).

In [ ]:
alpha = 0.05

shapiro_usou = stats.shapiro(usou)
shapiro_nao_usou = stats.shapiro(nao_usou)

normal_usou = shapiro_usou.pvalue >= alpha
normal_nao_usou = shapiro_nao_usou.pvalue >= alpha

print('--- Shapiro-Wilk: USOU CHATGPT ---')
print(f'Estatística W = {shapiro_usou.statistic:.4f}')
print(f'p-valor       = {shapiro_usou.pvalue:.4f}')
print(f'Normal (p >= {alpha})? {normal_usou}')
print()
print('--- Shapiro-Wilk: NÃO USOU CHATGPT ---')
print(f'Estatística W = {shapiro_nao_usou.statistic:.4f}')
print(f'p-valor       = {shapiro_nao_usou.pvalue:.4f}')
print(f'Normal (p >= {alpha})? {normal_nao_usou}')
print()

ambos_normais = normal_usou and normal_nao_usou
if ambos_normais:
    print('Ambos os grupos passaram no teste de normalidade.')
else:
    print('Ao menos um grupo não é normal.')

--- Shapiro-Wilk: USOU CHATGPT ---
Estatística W = 0.9212
p-valor       = 0.0622
Normal (p >= 0.05)? True

--- Shapiro-Wilk: NÃO USOU CHATGPT ---
Estatística W = 0.9526
p-valor       = 0.3088
Normal (p >= 0.05)? True

Ambos os grupos passaram no teste de normalidade.


### 5. Teste Mann-Whitney U

In [ ]:
alpha_mw = 0.05

stat_u, p_valor = stats.mannwhitneyu(usou, nao_usou, alternative='two-sided')

print(f'Estatística U = {stat_u:.4f}')
print(f'p-valor       = {p_valor:.4f}')
print()
if p_valor < alpha_mw:
    print(f' Diferença estatisticamente significativa (p < {alpha_mw}).')
else:
    print(f' Diferença NÃO significativa (p >= {alpha_mw}).')

Estatística U = 411.5000
p-valor       = 0.0109

 Diferença estatisticamente significativa (p < 0.05).


### 6. Tamanho do efeito - Cliff's Delta

Mede a magnitude da diferença entre os grupos independente do p-valor.  

Convenção de Romano et al. (2006):  
- |δ| < 0,147 → **negligível**  
- 0,147 ≤ |δ| < 0,330 → **pequeno**  
- 0,330 ≤ |δ| < 0,474 → **médio**  
- |δ| ≥ 0,474 → **grande**

In [ ]:
def cliffs_delta(group_a, group_b):
    """Calcula o Cliff's Delta entre dois grupos."""
    n_a, n_b = len(group_a), len(group_b)
    dominance = sum(
        1 if a > b else (-1 if a < b else 0)
        for a in group_a
        for b in group_b
    )
    delta = dominance / (n_a * n_b)
    return delta

def interpretar_delta(d):
    abs_d = abs(d)
    if abs_d < 0.147:
        return 'negligível'
    elif abs_d < 0.330:
        return 'pequeno'
    elif abs_d < 0.474:
        return 'médio'
    else:
        return 'grande'

delta = cliffs_delta(usou, nao_usou)
interpretacao = interpretar_delta(delta)

print(f"Cliff's Delta = {delta:.4f}")
print(f'Magnitude     = {interpretacao}')

Cliff's Delta = 0.4288
Magnitude     = médio


### 7. Tabela de resultados

In [ ]:
resultados = [
    {'Métrica': 'Total de participantes', 'Valor': f'{n_total}'},
    {'Métrica': 'Eliminados', 'Valor': f'{n_eliminados}'},
    {'Métrica': 'n — Usou ChatGPT', 'Valor': f'{n_usou}'},
    {'Métrica': 'n — Não usou ChatGPT', 'Valor': f'{n_nao_usou}'},
    {'Métrica': 'Média — Usou ChatGPT', 'Valor': f'{np.mean(usou):.4f}'},
    {'Métrica': 'Média — Não usou ChatGPT', 'Valor': f'{np.mean(nao_usou):.4f}'},
    {'Métrica': 'Variância — Usou ChatGPT', 'Valor': f'{np.var(usou, ddof=1):.4f}'},
    {'Métrica': 'Variância — Não usou ChatGPT', 'Valor': f'{np.var(nao_usou, ddof=1):.4f}'},
    {'Métrica': 'Estatística U (Mann-Whitney)', 'Valor': f'{stat_u:.4f}'},
    {'Métrica': 'p-valor (Mann-Whitney)', 'Valor': f'{p_valor:.4f}'},
    {'Métrica': "Cliff's Delta", 'Valor': f'{delta:.4f}'},
    {'Métrica': 'Magnitude do efeito', 'Valor': interpretacao},
]

In [ ]:
tabela = pd.DataFrame(resultados)

display(
    tabela.style
    .hide(axis='index')
    .set_properties(**{
        'text-align': 'left',
        'font-size': '13px'
    })
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('font-weight', 'bold'),
                ('text-align', 'left'),
                ('border-top', '1px solid black'),
                ('border-bottom', '1px solid black'),
                ('background-color', 'white'),
                ('color', 'black')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('border', 'none')
            ]
        },
        {
            'selector': 'table',
            'props': [
                ('border-collapse', 'collapse'),
                ('border-bottom', '1px solid black')
            ]
        }
    ])
)

Métrica,Valor
Total de participantes,57
Eliminados,9
n — Usou ChatGPT,24
n — Não usou ChatGPT,24
Média — Usou ChatGPT,6.5979
Média — Não usou ChatGPT,4.9792
Variância — Usou ChatGPT,3.4428
Variância — Não usou ChatGPT,4.1950
Estatística U (Mann-Whitney),411.5000
p-valor (Mann-Whitney),0.0109
